# 03 — Spring AI integration

`JavaVectorsVectorStore` implements the Spring AI [`VectorStore`](https://docs.spring.io/spring-ai/reference/api/vectordbs.html) contract, so any Spring AI retriever / RAG pipeline that talks to `VectorStore` can swap in `java-vectors` without code changes elsewhere.

This notebook skips the full Spring Boot context — we just exercise the vector-store adapter directly against a `VectorCollection`. For an end-to-end Boot app with an LLM in the loop, see `:demos:spring-ai-rag`.

**Note on dependencies.** Spring AI is a large stack; instead of pulling `spring-ai-core` via `%maven` we keep this notebook scoped to the pieces that `java-vectors` exposes directly. To run a full RAG, start from `:demos:spring-ai-rag`.

In [1]:
%classpath /home/jovyan/work/vectors/vectors-core/build/libs/vectors-core-0.1.0-SNAPSHOT.jar
%classpath /home/jovyan/work/vectors/vectors-storage/build/libs/vectors-storage-0.1.0-SNAPSHOT.jar
%classpath /home/jovyan/work/vectors/vectors-quantization/build/libs/vectors-quantization-0.1.0-SNAPSHOT.jar
%classpath /home/jovyan/work/vectors/vectors-hnsw/build/libs/vectors-hnsw-0.1.0-SNAPSHOT.jar
%classpath /home/jovyan/work/vectors/vectors-vamana/build/libs/vectors-vamana-0.1.0-SNAPSHOT.jar
%classpath /home/jovyan/work/vectors/vectors-ivf/build/libs/vectors-ivf-0.1.0-SNAPSHOT.jar
%classpath /home/jovyan/work/vectors/vectors-gpu/build/libs/vectors-gpu-0.1.0-SNAPSHOT.jar
%classpath /home/jovyan/work/vectors/vectors-db/build/libs/vectors-db-0.1.0-SNAPSHOT.jar
%classpath /home/jovyan/work/vectors/vectors-spring-ai/build/libs/vectors-spring-ai-0.1.0-SNAPSHOT.jar


In [2]:
%%loadFromPOM
<dependencies>
  <dependency>
    <groupId>org.apache.arrow</groupId>
    <artifactId>arrow-vector</artifactId>
    <version>19.0.0</version>
  </dependency>
  <dependency>
    <groupId>org.apache.arrow</groupId>
    <artifactId>arrow-memory-unsafe</artifactId>
    <version>19.0.0</version>
  </dependency>
  <dependency>
    <groupId>org.springframework.ai</groupId>
    <artifactId>spring-ai-vector-store</artifactId>
    <version>1.1.4</version>
  </dependency>
  <dependency>
    <groupId>org.springframework.ai</groupId>
    <artifactId>spring-ai-model</artifactId>
    <version>1.1.4</version>
  </dependency>
  <dependency>
    <groupId>io.micrometer</groupId>
    <artifactId>micrometer-observation</artifactId>
    <version>1.14.4</version>
  </dependency>
</dependencies>


In [3]:
import com.integrallis.vectors.core.SimilarityFunction;
import com.integrallis.vectors.db.IndexType;
import com.integrallis.vectors.db.VectorCollection;
import com.integrallis.vectors.spring.ai.JavaVectorsVectorStore;
import org.springframework.ai.document.Document;
import org.springframework.ai.embedding.Embedding;
import org.springframework.ai.embedding.EmbeddingModel;
import org.springframework.ai.embedding.EmbeddingRequest;
import org.springframework.ai.embedding.EmbeddingResponse;
import org.springframework.ai.vectorstore.SearchRequest;
import org.springframework.ai.vectorstore.VectorStore;

import java.util.*;

/** Deterministic trigram-hash embedder. Fine for notebooks; don't ship it. */
final class TrigramEmbeddingModel implements EmbeddingModel {
    private final int dim;
    TrigramEmbeddingModel(int dim) { this.dim = dim; }
    @Override public int dimensions() { return dim; }
    @Override public EmbeddingResponse call(EmbeddingRequest req) {
        List<Embedding> out = new ArrayList<>();
        int i = 0;
        for (String t : req.getInstructions()) out.add(new Embedding(trigram(t), i++));
        return new EmbeddingResponse(out);
    }
    @Override public float[] embed(String text) { return trigram(text); }
    @Override public float[] embed(Document doc) { return trigram(doc.getText()); }
    @Override public List<float[]> embed(List<String> txts) { return txts.stream().map(this::trigram).toList(); }
    private float[] trigram(String text) {
        float[] v = new float[dim];
        String s = text.toLowerCase();
        if (s.length() < 3) s = "  " + s + "  ";
        for (int i = 0; i <= s.length() - 3; i++) {
            int h = 31*(31*s.charAt(i) + s.charAt(i+1)) + s.charAt(i+2);
            v[Math.floorMod(h, dim)] += 1f;
        }
        float n = 0f; for (float x : v) n += x*x; n = (float)Math.sqrt(n);
        if (n > 0) for (int i = 0; i < dim; i++) v[i] /= n;
        return v;
    }
}

In [4]:
EmbeddingModel embeddingModel = new TrigramEmbeddingModel(128);

VectorCollection collection = VectorCollection.builder()
    .dimension(embeddingModel.dimensions())
    .metric(SimilarityFunction.COSINE)
    .indexType(IndexType.HNSW)
    .hnswM(16).hnswEfConstruction(100)
    .autoCommitThreshold(32)
    .build();

VectorStore store = JavaVectorsVectorStore.builder(embeddingModel, collection)
    .collectionName("notebook-03")
    .commitAfterAdd(true)
    .build();

List<Document> docs = List.of(
    new Document("HNSW is a graph-based approximate nearest neighbor index."),
    new Document("Product quantization compresses vectors into short codes."),
    new Document("mmap maps a file into the process address space for zero-copy reads."),
    new Document("SIMD via Panama Vector API accelerates distance kernels on the JVM."),
    new Document("Scalar int8 quantization gives 4x memory reduction with <1% recall loss."));
store.add(docs);

List<Document> matches = store.similaritySearch(
    SearchRequest.builder().query("What is SIMD?").topK(3).build());
for (Document m : matches) System.out.printf("  score=%.4f  %s%n", m.getScore(), m.getText());

SLF4J(W): No SLF4J providers were found.


SLF4J(W): Defaulting to no-operation (NOP) logger implementation


SLF4J(W): See https://www.slf4j.org/codes.html#noProviders for further details.


  score=

0.6119

...

0.6044

Product quantization compresses vectors into short codes.

  score=

0.5909

SIMD via Panama Vector API accelerates distance kernels on the JVM.

In [5]:
// Spring AI's VectorStore also supports delete by id and metadata filter queries —
// both flow through to the underlying VectorCollection via FilterConverter.
String targetId = docs.get(0).getId();
store.delete(List.of(targetId));
System.out.println("after delete, collection size: " + collection.size());

collection.close();

after delete, collection size: 4


## Next up

- **04 — LangChain4j:** the same idea on the other major Java LLM framework.
- **05 — Embedding cache:** wrap any `EmbeddingModel` (Spring AI or LangChain4j) with a cache.